# `ngllib` remote server — notebook

Spins up a real `ngllib.Environment` (Playwright + Chromium driving
Neuroglancer) and serves it over a transport so a `RemoteEnv` in another
notebook (or process) can drive it. Pair with
[`remote_client.ipynb`](./remote_client.ipynb).

**Prerequisites:**

- `ngllib` installed (`pip install -e .` from the repo root).
- Chromium downloaded: `playwright install chromium` (~450 MB, one-time).

**Why we run `serve` in a background thread:** Jupyter runs an asyncio
event loop on the kernel's main thread, and Playwright's *sync* API
(`sync_playwright()`) refuses to start inside one. Putting `serve` on a
fresh thread (which has no event loop) sidesteps that and is also nicer
UX — the cell returns immediately, the kernel stays responsive, and you
can run more cells in this notebook while the server is up.

**How to use:**

1. Run cells 1–3 of this notebook to set up the env + transport.
2. Run cell 4 — it spawns `serve` in a background thread and returns.
3. Open `remote_client.ipynb` and run its cells.
4. The server exits automatically when the client sends `env.close()`.
   (Or run cell 5 below to force a stop.)

In [ ]:
import threading

from ngllib import Environment
from ngllib.distributed.serve import serve
from ngllib.distributed.transports import SocketTransport, FilesystemTransport

## 1. Build the environment

Standard `ngllib.Environment`. `browser_restart_every=None` disables
the periodic browser restart so it doesn't interfere with this short demo;
leave it at the default `90` for real long-running training.

In [ ]:
env = Environment(
    headless=True,
    orientation="euler",
    browser_restart_every=None,
)
print("observation_space:", env.observation_space)
print("action_space:    ", env.action_space)

## 2. Build the transport

Default is a TCP socket on `127.0.0.1:5555`. Swap the comments to use
`FilesystemTransport` (file-swap on a shared scratch dir — handy for
firewalled clusters, or for inspecting messages on disk).

In [ ]:
HOST = "127.0.0.1"
PORT = 5555

transport = SocketTransport.server(host=HOST, port=PORT, timeout=600)
print(f"transport: tcp://{HOST}:{PORT}")

# --- Filesystem alternative ---
# import tempfile, os
# scratch = tempfile.mkdtemp(prefix="ngllib_remote_")
# ACTION_DIR = os.path.join(scratch, "actions")
# OBS_DIR    = os.path.join(scratch, "obs")
# os.makedirs(ACTION_DIR, exist_ok=True)
# os.makedirs(OBS_DIR,    exist_ok=True)
# print("client should use these dirs:")
# print(f"  ACTION_DIR = {ACTION_DIR!r}")
# print(f"  OBS_DIR    = {OBS_DIR!r}")
# transport = FilesystemTransport.server(
#     action_dir=ACTION_DIR, obs_dir=OBS_DIR,
#     timeout=600, cleanup_on_init=True,
# )

## 3. Start the server (background thread, non-blocking)

Spawns `serve` on a fresh thread so Playwright's sync API works
without colliding with Jupyter's asyncio loop. The cell returns
immediately; the server runs in the background until the client
calls `env.close()` (clean exit) or the kernel restarts.

**First `reset()` from the client launches Chromium** (~10–15 s);
subsequent steps run at typical Neuroglancer step latency
(~150–300 ms over socket).

In [ ]:
def _run_server():
    try:
        serve(env, transport, wire_image_format="jpeg")
    except Exception as e:
        print(f"[server thread] exited with: {type(e).__name__}: {e}")
    else:
        print("[server thread] exited cleanly")

server_thread = threading.Thread(target=_run_server, daemon=True, name="ngllib-server")
server_thread.start()
print(f"server thread started ({server_thread.name}); waiting for client...")
print("-> now run the cells in remote_client.ipynb")

## 4. (Optional) Force-stop the server

Normally you don't need this — the server exits when the client sends
`env.close()` (i.e. when you run the last cell of `remote_client.ipynb`).
But if the client disconnected uncleanly or you just want to stop the
server before running the client, this closes the transport which makes
`serve`'s `recv()` raise `ConnectionLost` and the thread exit.

In [ ]:
import contextlib
with contextlib.suppress(Exception):
    transport.close()
server_thread.join(timeout=5)
print(f"server thread alive: {server_thread.is_alive()}")